# From Qiskit samples to a molecular energy

A local H₂ walkthrough: circuit → noisy samples → particle-number filter → selected Hamiltonian → SQD recovery. All computations use local simulators. No token, Runtime service, or hardware submission is used. Energies include nuclear repulsion and are in hartree.

In [1]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'sqd_lab').is_dir())
sys.path.insert(0, str(root))
import numpy as np
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.primitives.containers import BitArray
from sqd_lab.chemistry import hydrogen_chain
from sqd_lab.sampling import pair_circuit, sample_circuit, valid_mask
from sqd_lab.subspace import selected_energy
from qiskit_addon_sqd.fermion import diagonalize_fermionic_hamiltonian

## 1. Read a Qiskit measurement
The reference sampler simulates an ideal state and samples its measured bitstrings. A Bell state should produce only `00` and `11`; finite-shot counts fluctuate.

In [2]:
bell = QuantumCircuit(2)
bell.h(0); bell.cx(0, 1); bell.measure_all()
counts = StatevectorSampler(seed=42).run([bell], shots=128).result()[0].data.meas.get_counts()
assert sum(counts.values()) == 128 and set(counts) <= {'00', '11'}
print(counts)

{'11': 60, '00': 68}


## 2. Fix the chemistry before sampling
H₂ at 0.735 Å in STO-3G has two spatial orbitals and four spin orbitals. PySCF supplies molecular integrals and an independent full configuration-interaction (FCI) reference. Qiskit Nature maps the electronic Hamiltonian to four qubits. The reference is an evaluation target, never an input to the sampling circuit.

In [3]:
molecule = hydrogen_chain(2, .735)
print(f'HF:  {molecule.hf_energy:.10f} Ha')
print(f'FCI: {molecule.exact_energy:.10f} Ha')
print(f'Spin occupations: {molecule.nelec}; spatial orbitals: {molecule.norb}')

HF:  -1.1169989968 Ha
FCI: -1.1373060358 Ha
Spin occupations: (1, 1); spatial orbitals: 2


## 3. Prepare a simple correlated state
The fixed pair circuit conserves one alpha and one beta electron. It is a teaching ansatz, without variational optimization. Qiskit prints high-index wires first: beta bits are on the left, alpha bits on the right.

In [4]:
circuit = pair_circuit(molecule.norb, theta=.2)
print(circuit.draw(output='text'))

     ┌──────────┐               ┌───┐
q_0: ┤ Ry(-0.4) ├──■────■────■──┤ X ├
     └──────────┘┌─┴─┐  │    │  └───┘
q_1: ────────────┤ X ├──┼────┼───────
                 └───┘┌─┴─┐  │  ┌───┐
q_2: ─────────────────┤ X ├──┼──┤ X ├
                      └───┘┌─┴─┐└───┘
q_3: ──────────────────────┤ X ├─────
                           └───┘     


## 4. Sample and check a physical constraint
We explicitly add independent synthetic readout bit flips with probability 0.08 per bit. This is not measured device noise. Particle-number filtering rejects invalid counts; errors that preserve particle number can remain.

In [5]:
raw = sample_circuit(circuit, shots=128, seed=1201, readout_error=.08)
filtered = raw[valid_mask(raw, molecule.nelec)]
print(f'Accepted: {len(filtered)}/{len(raw)}')
print(f'Distinct configurations: {len(np.unique(filtered, axis=0))}')
counts = {}
for row in raw:
    bitstring = ''.join('1' if bit else '0' for bit in row)
    counts[bitstring] = counts.get(bitstring, 0) + 1
assert sum(counts.values()) == 128
print(dict(sorted(counts.items())))

Accepted: 95/128
Distinct configurations: 3
{'0000': 2, '0001': 9, '0011': 2, '0100': 5, '0101': 90, '0111': 6, '1001': 2, '1010': 3, '1101': 8, '1111': 1}


## 5. QSCI: diagonalize a selected basis
Distinct observed determinants define the subspace. Their measured probabilities do not become the eigenvector amplitudes: classical diagonalization finds those amplitudes. An independently checked projected Hamiltonian gives a variational upper bound to the fixed-sector FCI energy.

In [6]:
qsci = selected_energy(filtered, molecule.hamiltonian, molecule.offset)
assert qsci['energy'] >= molecule.exact_energy - 1e-8
print(f"Selected dimension: {qsci['dimension']}")
print(f"QSCI: {qsci['energy']:.10f} Ha")
print(f"Error: {1000*(qsci['energy']-molecule.exact_energy):.6f} mHa")

Selected dimension: 3
QSCI: -1.1373060358 Ha
Error: -0.000000 mHa


## 6. SQD: recover configurations iteratively
The official SQD add-on uses occupation estimates to recover particle-number-invalid samples and updates these estimates through diagonalization. Here each spin-sector string set is capped at two, so its Cartesian basis can contain at most four configurations. Recovery and basis expansion both affect the answer.

In [7]:
history = []
def callback(batches):
    history.append([float(result.energy + molecule.offset) for result in batches])
recovered = diagonalize_fermionic_hamiltonian(
    molecule.hcore, molecule.eri, BitArray.from_counts(counts),
    samples_per_batch=32, norb=molecule.norb, nelec=molecule.nelec,
    max_iterations=4, max_dim=2, symmetrize_spin=True, seed=1201, callback=callback)
energy = float(recovered.energy + molecule.offset)
assert energy >= molecule.exact_energy - 1e-8
print(f'SQD: {energy:.10f} Ha')
print(f'Basis dimension: {np.prod(recovered.sci_state.amplitudes.shape)}')
print('Iteration energies:', history)

SQD: -1.1373060358 Ha
Basis dimension: 4
Iteration energies: [[-1.1373060357534004], [-1.1373060357534004]]


## 7. Interpret the result
Exact agreement on H₂ is a correctness check, not quantum advantage. Its entire physical sector contains only four determinants. Uniform classical sampling can cover it too. The [H₄ benchmark and report](../docs/sqd/REPORT.md) show what changes when the circuit has limited support and the recovered subspace is capped.

Run the reproducible sweep from the repository root: `uv run python -m sqd_lab --config configs/sqd/h4.json --output artifacts/sqd/my-h4-run`. Use a fresh directory.

Sources: [SQD quickstart](https://qiskit.github.io/qiskit-addon-sqd/guides/quickstart.html), [QSCI](https://arxiv.org/abs/2302.11320), [Qiskit reference primitives](https://quantum.cloud.ibm.com/docs/en/guides/simulate-with-qiskit-sdk-primitives).